# E-commerce Checkout A/B Test
## Day 14 — Secondary and Guardrail Metrics

### Executive Summary

Day 12 established a statistically significant **+2.16 percentage-point** primary conversion lift. This supporting analysis evaluates whether that improvement is accompanied by favorable secondary outcomes or guardrail harm.

Treatment produced higher retained revenue per exposed user (**$24.52 → $26.33**) and a shorter median time to purchase (**28.13 → 26.82 minutes**). Payment failure was slightly lower. Checkout error and refund/cancellation rates were slightly higher, although their confidence intervals include zero.

Because failure to reject a harm hypothesis is not proof of safety—and no non-inferiority margins were pre-specified—the treatment remains a **conditional launch candidate**. The appropriate Day 14 decision is **Need more data before full launch**.

### Metric Definitions

| Metric | Role | Numerator / value | Denominator / population | Window |
|---|---|---|---|---|
| Median checkout completion time | Secondary | Minutes from first valid exposure to attributed purchase | Converted mature exposed users | Within 24 hours |
| Payment success rate | Secondary | Payment-attempt users without a recorded payment failure | Users with a qualifying payment attempt | Within 24 hours |
| Retained revenue per exposed user | Secondary | Revenue from attributed orders with final status `completed` | Mature exposed users | Purchase plus 7-day order follow-up |
| Payment failure rate | Guardrail | Payment-attempt users with a recorded failure | Users with a qualifying payment attempt | Within 24 hours |
| Checkout error rate | Guardrail | Users with an error in the first valid exposure session | Mature exposed users | Exposure session |
| Refund/cancellation rate | Guardrail | Attributed purchasers with final status `refunded` or `cancelled` | Attributed purchasers with complete follow-up | 7 days after purchase |

The data contain `payment_failure` but no separate `payment_success` event, so payment success is operationalized as one minus the user-level payment failure rate among payment-attempt users.

In [1]:
from pathlib import Path
from math import sqrt
from statistics import NormalDist
import sqlite3

import pandas as pd
from scipy.stats import t as student_t

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = None

for path in [START_DIR, *START_DIR.parents]:
    if (
        path.name == "week2_checkout_experiment"
        and (path / "data" / "raw").is_dir()
    ):
        PROJECT_ROOT = path
        break

    candidate = path / "week2_checkout_experiment"
    if (candidate / "data" / "raw").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the week2_checkout_experiment directory."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
FUNNEL_SQL_PATH = PROJECT_ROOT / "sql" / "02_funnel_analysis.sql"
ORDER_DATA_CUTOFF = "2026-07-22 23:59:59"

print("Project root:", PROJECT_ROOT.name)
print("Reused SQL definition:", FUNNEL_SQL_PATH.relative_to(PROJECT_ROOT))

Project root: week2_checkout_experiment
Reused SQL definition: sql/02_funnel_analysis.sql


## 1. Build User-Level Supporting Metrics

The unchanged Day 11 SQL first reconstructs the eligible mature exposed-user population and ordered funnel. Supporting flags are then attached at the same one-row-per-user grain.

In [2]:
conn = sqlite3.connect(":memory:")

for table_name in ["users", "experiment_assignments", "events", "orders"]:
    table = pd.read_csv(RAW_DATA_DIR / f"{table_name}.csv")
    table.to_sql(table_name, conn, index=False, if_exists="replace")

conn.executescript(FUNNEL_SQL_PATH.read_text(encoding="utf-8"))

supporting_sql = f"""
CREATE TEMP TABLE exposure_sessions AS
SELECT
    f.user_id,
    MIN(e.session_id) AS exposure_session_id
FROM user_level_funnel f
JOIN deduplicated_events e
    ON f.user_id = e.user_id
   AND f.experiment_group = e.experiment_group
   AND e.event_name = 'checkout_view'
   AND e.event_timestamp = f.exposure_timestamp
GROUP BY f.user_id;

CREATE TEMP TABLE user_guardrail_flags AS
SELECT
    f.user_id,
    MAX(
        CASE
            WHEN e.event_name = 'payment_failure'
             AND f.payment_attempt_timestamp IS NOT NULL
             AND datetime(e.event_timestamp)
                 >= datetime(f.payment_attempt_timestamp)
             AND datetime(e.event_timestamp)
                 <= datetime(f.exposure_timestamp, '+24 hours')
            THEN 1 ELSE 0
        END
    ) AS had_payment_failure,
    MAX(
        CASE
            WHEN e.event_name = 'checkout_error'
             AND e.session_id = s.exposure_session_id
             AND datetime(e.event_timestamp)
                 >= datetime(f.exposure_timestamp)
            THEN 1 ELSE 0
        END
    ) AS had_checkout_error
FROM user_level_funnel f
JOIN exposure_sessions s
    ON f.user_id = s.user_id
LEFT JOIN deduplicated_events e
    ON f.user_id = e.user_id
   AND f.experiment_group = e.experiment_group
GROUP BY f.user_id;
"""

conn.executescript(supporting_sql)

user_metrics = pd.read_sql_query(
    f"""
    SELECT
        f.*,
        g.had_payment_failure,
        g.had_checkout_error,
        CASE
            WHEN f.reached_payment_attempt = 1
            THEN 1 - g.had_payment_failure
        END AS payment_succeeded,
        CASE
            WHEN f.purchase_timestamp IS NOT NULL
            THEN (
                julianday(f.purchase_timestamp)
                - julianday(f.exposure_timestamp)
            ) * 24 * 60
        END AS completion_minutes,
        o.order_id,
        o.order_amount,
        o.order_status,
        CASE
            WHEN f.purchase_timestamp IS NOT NULL
             AND datetime(f.purchase_timestamp, '+7 days')
                 <= datetime('{ORDER_DATA_CUTOFF}')
            THEN 1 ELSE 0
        END AS mature_order_followup,
        CASE
            WHEN o.order_status = 'completed' THEN o.order_amount
            ELSE 0
        END AS retained_revenue,
        CASE
            WHEN o.order_status IN ('refunded', 'cancelled') THEN 1
            ELSE 0
        END AS refunded_or_cancelled
    FROM user_level_funnel f
    LEFT JOIN user_guardrail_flags g
        ON f.user_id = g.user_id
    LEFT JOIN orders o
        ON f.user_id = o.user_id
       AND f.experiment_group = o.experiment_group
       AND f.purchase_timestamp = o.purchase_timestamp
    ORDER BY f.user_id;
    """,
    conn,
)

In [3]:
assert len(user_metrics) == 15_467
assert user_metrics["user_id"].nunique() == len(user_metrics)
assert user_metrics["reached_purchase"].sum() == 4_415
assert user_metrics["order_id"].notna().sum() == 4_415
assert user_metrics.loc[
    user_metrics["reached_purchase"].eq(1),
    "mature_order_followup",
].eq(1).all()

validation_summary = pd.DataFrame(
    {
        "check": [
            "analysis rows",
            "unique users",
            "attributed purchasers",
            "matched attributed orders",
            "purchasers with mature 7-day follow-up",
        ],
        "value": [15_467, 15_467, 4_415, 4_415, 4_415],
    }
)
print(validation_summary.to_string(index=False))

                                 check  value
                         analysis rows  15467
                          unique users  15467
                 attributed purchasers   4415
             matched attributed orders   4415
purchasers with mature 7-day follow-up   4415


## 2. Metric Results by Experiment Group

In [4]:
summary_rows = []
for experiment_group, group_data in user_metrics.groupby("experiment_group"):
    attempted = group_data.loc[group_data["reached_payment_attempt"].eq(1)]
    purchasers = group_data.loc[group_data["reached_purchase"].eq(1)]
    mature_purchasers = purchasers.loc[purchasers["mature_order_followup"].eq(1)]

    summary_rows.append(
        {
            "experiment_group": experiment_group,
            "exposed_users": len(group_data),
            "payment_attempt_users": len(attempted),
            "purchase_users": len(purchasers),
            "payment_failure_users": int(attempted["had_payment_failure"].sum()),
            "payment_failure_rate": attempted["had_payment_failure"].mean(),
            "payment_success_rate": attempted["payment_succeeded"].mean(),
            "checkout_error_users": int(group_data["had_checkout_error"].sum()),
            "checkout_error_rate": group_data["had_checkout_error"].mean(),
            "refund_cancel_users": int(mature_purchasers["refunded_or_cancelled"].sum()),
            "refund_cancel_rate": mature_purchasers["refunded_or_cancelled"].mean(),
            "retained_revenue_per_exposed": group_data["retained_revenue"].mean(),
            "median_completion_minutes": purchasers["completion_minutes"].median(),
        }
    )

metric_summary = pd.DataFrame(summary_rows).sort_values("experiment_group")
metric_summary_display = metric_summary[
    [
        "experiment_group",
        "payment_success_rate",
        "payment_failure_rate",
        "checkout_error_rate",
        "refund_cancel_rate",
        "retained_revenue_per_exposed",
        "median_completion_minutes",
    ]
].copy()

for column in [
    "payment_success_rate",
    "payment_failure_rate",
    "checkout_error_rate",
    "refund_cancel_rate",
]:
    metric_summary_display[column] = metric_summary_display[column].map(
        "{:.2%}".format
    )

metric_summary_display["retained_revenue_per_exposed"] = metric_summary_display[
    "retained_revenue_per_exposed"
].map(lambda value: f"${value:.2f}")
metric_summary_display["median_completion_minutes"] = metric_summary_display[
    "median_completion_minutes"
].map(lambda value: f"{value:.2f}")

print(metric_summary_display.to_string(index=False))

experiment_group payment_success_rate payment_failure_rate checkout_error_rate refund_cancel_rate retained_revenue_per_exposed median_completion_minutes
         control               94.64%                5.36%               3.07%              5.68%                       $24.52                     28.13
       treatment               94.87%                5.13%               3.49%              6.48%                       $26.33                     26.82


## 3. Guardrail Differences and Confidence Intervals

The tests below are two-sided descriptive checks. They are not non-inferiority tests because acceptable harm margins were not defined before the experiment.

In [5]:
normal = NormalDist()
z_critical = normal.inv_cdf(0.975)
summary_by_group = metric_summary.set_index("experiment_group")

def binary_rate_difference(metric, denominator_column, numerator_column):
    control = summary_by_group.loc["control"]
    treatment = summary_by_group.loc["treatment"]

    n_control = int(control[denominator_column])
    x_control = int(control[numerator_column])
    n_treatment = int(treatment[denominator_column])
    x_treatment = int(treatment[numerator_column])

    p_control = x_control / n_control
    p_treatment = x_treatment / n_treatment
    difference = p_treatment - p_control
    pooled_rate = (x_control + x_treatment) / (n_control + n_treatment)
    pooled_se = sqrt(
        pooled_rate * (1 - pooled_rate) * (1 / n_control + 1 / n_treatment)
    )
    unpooled_se = sqrt(
        p_control * (1 - p_control) / n_control
        + p_treatment * (1 - p_treatment) / n_treatment
    )
    z_statistic = difference / pooled_se
    p_value = 2 * normal.cdf(-abs(z_statistic))

    return {
        "metric": metric,
        "control_rate": p_control,
        "treatment_rate": p_treatment,
        "difference": difference,
        "ci_lower": difference - z_critical * unpooled_se,
        "ci_upper": difference + z_critical * unpooled_se,
        "p_value": p_value,
    }

guardrail_results = pd.DataFrame(
    [
        binary_rate_difference(
            "Payment failure",
            "payment_attempt_users",
            "payment_failure_users",
        ),
        binary_rate_difference(
            "Checkout error",
            "exposed_users",
            "checkout_error_users",
        ),
        binary_rate_difference(
            "Refund/cancellation",
            "purchase_users",
            "refund_cancel_users",
        ),
    ]
)

guardrail_display = guardrail_results.copy()
guardrail_display["control_rate"] = guardrail_display["control_rate"].map(
    "{:.2%}".format
)
guardrail_display["treatment_rate"] = guardrail_display["treatment_rate"].map(
    "{:.2%}".format
)
guardrail_display["difference"] = guardrail_display["difference"].map(
    lambda value: f"{value * 100:+.2f} pp"
)
guardrail_display["95% CI"] = [
    f"[{lower * 100:+.2f}, {upper * 100:+.2f}] pp"
    for lower, upper in zip(
        guardrail_display.pop("ci_lower"),
        guardrail_display.pop("ci_upper"),
    )
]
guardrail_display["p_value"] = guardrail_display["p_value"].map("{:.4f}".format)

print(guardrail_display.to_string(index=False))

             metric control_rate treatment_rate difference p_value            95% CI
    Payment failure        5.36%          5.13%   -0.23 pp  0.5810 [-1.03, +0.58] pp
     Checkout error        3.07%          3.49%   +0.42 pp  0.1441 [-0.14, +0.98] pp
Refund/cancellation        5.68%          6.48%   +0.80 pp  0.2690 [-0.61, +2.21] pp


## 4. Retained Revenue per Exposed User

Each mature exposed user contributes final completed-order revenue or zero. A Welch comparison is used because the user-level revenue distribution is zero-inflated and group variances need not be equal. With this large sample, the interval is a useful supporting approximation; the metric remains secondary.

In [6]:
control_revenue = user_metrics.loc[
    user_metrics["experiment_group"].eq("control"),
    "retained_revenue",
]
treatment_revenue = user_metrics.loc[
    user_metrics["experiment_group"].eq("treatment"),
    "retained_revenue",
]

control_mean = control_revenue.mean()
treatment_mean = treatment_revenue.mean()
revenue_difference = treatment_mean - control_mean
control_variance = control_revenue.var(ddof=1)
treatment_variance = treatment_revenue.var(ddof=1)
n_control = len(control_revenue)
n_treatment = len(treatment_revenue)
revenue_se = sqrt(
    control_variance / n_control
    + treatment_variance / n_treatment
)
welch_df = (
    control_variance / n_control + treatment_variance / n_treatment
) ** 2 / (
    (control_variance / n_control) ** 2 / (n_control - 1)
    + (treatment_variance / n_treatment) ** 2 / (n_treatment - 1)
)
revenue_t_statistic = revenue_difference / revenue_se
revenue_p_value = 2 * student_t.sf(abs(revenue_t_statistic), welch_df)
revenue_t_critical = student_t.ppf(0.975, welch_df)
revenue_ci_lower = revenue_difference - revenue_t_critical * revenue_se
revenue_ci_upper = revenue_difference + revenue_t_critical * revenue_se

revenue_summary = pd.DataFrame(
    {
        "control_rpeu": [f"${control_mean:.2f}"],
        "treatment_rpeu": [f"${treatment_mean:.2f}"],
        "absolute_difference": [f"${revenue_difference:+.2f}"],
        "relative_lift": [f"{revenue_difference / control_mean:+.2%}"],
        "95% CI": [f"[${revenue_ci_lower:+.2f}, ${revenue_ci_upper:+.2f}]"],
        "p_value": [f"{revenue_p_value:.4f}"],
    }
)
print(revenue_summary.to_string(index=False))

control_rpeu treatment_rpeu absolute_difference relative_lift           95% CI p_value
      $24.52         $26.33              $+1.81        +7.40% [$+0.30, $+3.33]  0.0192


## Interpretation and Decision

- **Payment reliability:** Failure decreased from 5.36% to 5.13% (**-0.23 pp**, 95% CI **[-1.03, +0.58] pp**). The estimate is favorable, but the interval includes a small increase.
- **Checkout stability:** Error rate increased from 3.07% to 3.49% (**+0.42 pp**, 95% CI **[-0.14, +0.98] pp**). The test is not significant, but the upper bound does not rule out meaningful harm.
- **Order quality:** Refund/cancellation increased from 5.68% to 6.48% (**+0.80 pp**, 95% CI **[-0.61, +2.21] pp**). Again, non-significance is not evidence of equivalence.
- **Commercial value:** Retained revenue per exposed user increased by **$1.81** (**+7.40%**), with a 95% CI of **+$0.30 to +$3.33**.
- **Speed:** Median completion time decreased by **1.31 minutes** among purchasers. This is descriptive because conversion conditions membership in this secondary metric.

### Day 14 Recommendation

**Need more data before full launch.** The primary conversion effect and revenue signal are positive, but the experiment was not designed with guardrail non-inferiority margins, and the checkout-error and refund/cancellation intervals still permit practically relevant harm. A follow-up should pre-specify acceptable guardrail deltas and collect enough data to rule them out.

In [7]:
conn.close()
print("In-memory SQLite connection closed.")

In-memory SQLite connection closed.
